In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
metrics_data = pd.read_csv('../datasets/processed/combined_metrics.csv')
metrics_data['review_duration'] = metrics_data['review_duration'] / 3600  # seconds → hours
agent_metrics = metrics_data[metrics_data['agent'].notna()].copy()
agent_metrics = agent_metrics[~agent_metrics['agent'].astype(str).str.fullmatch('human', case=False)]

# RQ2 compares agents using exactly the same metrics as RQ1: the 13
# continuous Table III measures plus the binary acceptance outcome.
RQ1_CONTINUOUS_METRICS = [
    'reviewers', 'review_comments', 'comments', 'commits',
    'changed_files', 'initial_workflow_files', 'revision_workflow_files',
    'review_duration', 'length_of_discussion', 'length_of_description',
    'code_churn', 'workflow_code_churn', 'build_failure_rate',
]
RQ1_BINARY_OUTCOMES = ['accepted']
metric_columns = RQ1_CONTINUOUS_METRICS + RQ1_BINARY_OUTCOMES
missing_rq1_metrics = [m for m in metric_columns if m not in agent_metrics.columns]
assert not missing_rq1_metrics, f'Missing RQ1 metrics: {missing_rq1_metrics}'
assert len(metric_columns) == 14, f'Expected 14 RQ1 metrics, found {len(metric_columns)}'

agent_counts = agent_metrics['agent'].value_counts().rename_axis('agent').reset_index(name='pr_count')
agent_order = agent_counts['agent'].tolist()

print(f'Agents included: {", ".join(agent_order)}')
print(f'Number of included metrics: {len(metric_columns)}')
print(metric_columns)


# Ai agents for each metrics overview

# Cross-agent metrics (AI agents only)

In [ ]:
# Keep only the named AI agents in RQ2.
agent_metrics_all = agent_metrics
agent_order_all = agent_order

agent_short = {
    'OpenAI_Codex': 'Codex', 'Devin': 'Devin', 'Cursor': 'Cursor',
    'Claude_Code': 'Claude', 'Copilot': 'Copilot',
}
agent_ns = {a: int((agent_metrics_all['agent'] == a).sum()) for a in agent_order_all}
group_colors = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2"]
agent_palette = dict(zip(agent_order_all, group_colors))

print('Groups included:', ', '.join(f'{agent_short[a]} (n={agent_ns[a]})' for a in agent_order_all))

In [ ]:
# Median-only metrics per AI agent. `accepted` is a
# binary (0/1) outcome, so its median would hide variation between groups;
# report its rate (proportion of accepted PRs) instead. Every continuous
# RQ1 metric uses the median across PRs.
BINARY_RATE_METRICS = {'accepted'}

def central_value(series, metric):
    return series.mean() if metric in BINARY_RATE_METRICS else series.median()

summary_table = pd.DataFrame({
    a: {m: central_value(agent_metrics_all.loc[agent_metrics_all['agent'] == a, m].dropna(), m)
        for m in metric_columns}
    for a in agent_order_all
}).reindex(columns=agent_order_all)

summary_table.columns = [f'{agent_short.get(a, a)} (n={agent_ns[a]})' for a in summary_table.columns]
summary_table.index.name = 'metric'
summary_table = summary_table.round(2)

summary_table.to_csv('../datasets/processed/rq2/agent_median_metrics.csv')


# Plot cross agent metric

In [ ]:
# Plot: one panel per RQ1 metric, one box (or bar for acceptance rate)
# per agent, median annotated above each box/bar.
import math
import matplotlib.patches as mpatches

METRIC_ORDER = metric_columns.copy()
assert METRIC_ORDER == RQ1_CONTINUOUS_METRICS + RQ1_BINARY_OUTCOMES

metric_labels = {
    'reviewers': 'Number of reviewers',
    'review_comments': 'Number of review comments',
    'comments': 'Number of comments',
    'commits': 'Number of commits',
    'changed_files': 'Number of total changed files',
    'initial_workflow_files': 'Number of workflow files change (first commit)',
    'revision_workflow_files': 'Number of workflow files change (revision commits)',
    'review_duration': 'Review duration (hours)',
    'length_of_discussion': 'Length of discussion (characters)',
    'length_of_description': 'Length of description (characters)',
    'code_churn': 'Total code churn',
    'workflow_code_churn': 'Code churn (workflow files)',
    'accepted': 'Acceptance rate',
    'build_failure_rate': 'Build failure rate',
}
CONTINUOUS_RATE_METRICS = {'build_failure_rate'}
RATE_METRICS = BINARY_RATE_METRICS | CONTINUOUS_RATE_METRICS
CONTINUOUS_METRICS = {'review_duration'}

def metric_axis_label(m):
    label = metric_labels.get(m, m.replace('_', ' ').title())
    return label

def fmt(val, metric):
    if pd.isna(val):
        return ''
    if metric in RATE_METRICS:
        return f'{val:.2f}'
    if metric in CONTINUOUS_METRICS:
        return f'{val:,.1f}' if val < 10 else f'{val:,.0f}'
    if val < 10:
        return f'{val:,.0f}' if float(val) == int(val) else f'{val:,.1f}'
    return f'{val:,.0f}'

def wrap_label_two_lines(text):
    """Break a label into two lines at the space closest to its midpoint."""
    mid = len(text) // 2
    left = text.rfind(' ', 0, mid)
    right = text.find(' ', mid)
    if left == -1 and right == -1:
        return text
    if left == -1:
        split_at = right
    elif right == -1:
        split_at = left
    else:
        split_at = left if (mid - left) <= (right - mid) else right
    return text[:split_at] + '\n' + text[split_at + 1:]

ncols = 4
nrows = math.ceil(len(METRIC_ORDER) / ncols)
n_ag = len(agent_order_all)
fig = plt.figure(figsize=(10.5, 2.15 * nrows), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.015, h_pad=0.015, wspace=0.035, hspace=0.05)
grid = fig.add_gridspec(nrows, ncols * 2)
last_row_count = len(METRIC_ORDER) % ncols or ncols
axes = []
for index in range(len(METRIC_ORDER)):
    row, column = divmod(index, ncols)
    if row == nrows - 1 and last_row_count < ncols:
        start = (ncols * 2 - last_row_count * 2) // 2 + column * 2
    else:
        start = column * 2
    axes.append(fig.add_subplot(grid[row, start:start + 2]))

ylabel_texts = {}
for ax, metric in zip(axes, METRIC_ORDER):
    if metric in BINARY_RATE_METRICS:
        rates = [agent_metrics_all.loc[agent_metrics_all['agent'] == a, metric].dropna().mean()
                 for a in agent_order_all]
        ax.bar(range(n_ag), rates, color=group_colors, edgecolor='#333333',
               linewidth=0.9, alpha=0.9, width=0.62)
        ymax = max([r for r in rates if pd.notna(r)] + [0]) or 1
        ax.set_ylim(0, ymax * 1.42)
        for i, r in enumerate(rates):
            label_gap = ymax * (0.03 + 0.09 * (i % 2))
            x_offset = -0.12 if i % 2 == 0 else 0.12
            ax.text(i + x_offset, r + label_gap, fmt(r, metric), ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold')
    else:
        data, medians = [], []
        for a in agent_order_all:
            s = agent_metrics_all.loc[agent_metrics_all['agent'] == a, metric].dropna()
            data.append(s.values if len(s) else [0.0])
            medians.append(s.median() if len(s) else 0.0)
        bp = ax.boxplot(data, positions=range(n_ag), widths=0.62, patch_artist=True, showfliers=False,
                        medianprops=dict(color='#1a1a1a', linewidth=1.6),
                        whiskerprops=dict(color='#444444', linewidth=1.0),
                        capprops=dict(color='#444444', linewidth=1.0),
                        boxprops=dict(edgecolor='#333333', linewidth=1.0))
        for i, patch in enumerate(bp['boxes']):
            patch.set_facecolor(group_colors[i])
            patch.set_alpha(0.9)
        caps = [bp['caps'][2 * i + 1].get_ydata()[0] for i in range(n_ag)]
        ymax = max(caps + [0]) or 1
        headroom = 1.42 if metric in RATE_METRICS else 1.24
        ax.set_ylim(0, ymax * headroom)
        ax.set_xlim(-0.5, n_ag - 0.5)
        for i, med in enumerate(medians):
            label_gap = ymax * (0.025 + 0.105 * (i % 2))
            x_offset = -0.12 if i % 2 == 0 else 0.12
            ax.text(i + x_offset, caps[i] + label_gap, fmt(med, metric), ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold')
    ylabel_texts[ax] = ax.set_ylabel(
        metric_axis_label(metric), fontsize=10.5, fontweight='bold', labelpad=2
    )
    # Agent identity is shown only in the shared legend.
    ax.set_xticks([])
    ax.set_xlabel('')
    ax.tick_params(axis='y', labelsize=9.5, pad=2)
    for tick_label in ax.get_yticklabels():
        tick_label.set_fontweight('bold')
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.set_axisbelow(True)
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_linewidth(0.8)

# Wrap only y-axis labels that would exceed their panel height.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for ax, label_text in ylabel_texts.items():
    if label_text.get_window_extent(renderer=renderer).height > ax.get_window_extent(renderer=renderer).height:
        label_text.set_text(wrap_label_two_lines(label_text.get_text()))

legend_handles = [mpatches.Patch(facecolor=agent_palette[a], edgecolor='#333333',
                                  label=agent_short.get(a, a)) for a in agent_order_all]
fig.legend(handles=legend_handles, loc='outside upper center', ncol=n_ag, frameon=False,
           prop={'size': 12.5, 'weight': 'bold'})

fig.savefig('../figures/rq2_agent_all_metrics.pdf', bbox_inches='tight')
plt.show()